In [ ]:
import pandas as pd

# --- 1. SETUP: File path and map center ---
csv_path = "D:/Project_P1/data/ebird_observations.csv"
df = pd.read_csv(csv_path, nrows=0)
# Print column names
print(df.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
import geopandas as gpd # <-- IMPORT GEOPANDAS

# --- 1. SETUP: Parameters and File Paths ---
CSV_PATH = "D:/Project_P1/data/ebird_observations.csv"
GPKG_PATH = "coimbatore_landUse.gpkg" # <-- CHANGED TO YOUR GPKG FILE
AIRPORT_COORDS = (11.0296, 77.0434)
OUTPUT_HTML_FILE = "coimbatore_bird_heatmap_optimized.html"
RADIUS_KM = 10
TOP_N_SPECIES = 5

# --- 2. DATA PROCESSING FUNCTIONS (Unchanged) ---

def load_and_clean_data(csv_path: str) -> pd.DataFrame:
    """Loads bird observation data, checks columns, and drops invalid rows."""
    print(f"Reading bird observation data from: {csv_path}")
    try:
        df = pd.read_csv(csv_path)
        required_columns = ['LATITUDE', 'LONGITUDE', 'COMMON NAME', 'OBSERVATION COUNT']
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"CSV must contain: {required_columns}")
        df.dropna(subset=['LATITUDE', 'LONGITUDE'], inplace=True)
        df['LATITUDE'] = pd.to_numeric(df['LATITUDE'], errors='coerce')
        df['LONGITUDE'] = pd.to_numeric(df['LONGITUDE'], errors='coerce')
        df['OBSERVATION COUNT'] = pd.to_numeric(df['OBSERVATION COUNT'], errors='coerce').fillna(1)
        df.dropna(subset=['LATITUDE', 'LONGITUDE', 'OBSERVATION COUNT'], inplace=True)
        print(f"Successfully loaded and cleaned {len(df)} observations.")
        return df
    except FileNotFoundError:
        print(f"Error: File not found at {csv_path}")
        exit()
    except Exception as e:
        print(f"An error occurred while reading the CSV: {e}")
        exit()

def vectorized_haversine(lats, lons, center_lat, center_lon):
    """Calculates great-circle distance in a vectorized way."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lats, lons, center_lat, center_lon])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# --- 3. MAP CREATION FUNCTIONS ---

def create_base_map(center_coords, radius_km):
    """Creates the base Folium map with airport marker and radius circle."""
    print("Creating base map...")
    base_map = folium.Map(location=center_coords, zoom_start=12)
    folium.Marker(
        location=center_coords,
        popup=f"Coimbatore Int'l Airport ({radius_km}km Radius)",
        icon=folium.Icon(color="darkblue", icon="plane", prefix="fa")
    ).add_to(base_map)
    folium.Circle(
        location=center_coords, radius=radius_km * 1000,
        color='darkblue', weight=2, dash_array='5, 5', fill=False,
        popup=f"{radius_km} km Radius"
    ).add_to(base_map)
    return base_map

# v-- THIS FUNCTION IS MODIFIED TO READ THE .GPKG FILE --v
def add_landuse_layer(folium_map, gpkg_path):
    """
    Loads digitized land use polygons from a GeoPackage file and adds them to the map,
    styling each polygon based on its 'use_type' property.
    """
    print(f"Adding land use layer from: {gpkg_path}")
    
    color_map = {
        'Waterbody': '#007BFF',
        'Agriculture': '#28A745',
        'Reserved Forest': '#206131',
        'Industrial': '#6F42C1',
        'Residential': '#FFC107',
        'Commercial': '#DC3545',
        'Default': '#6C757D'
    }

    def style_function(feature):
        use_type = feature['properties'].get('use_type', 'Default')
        return {
            'fillColor': color_map.get(use_type, color_map['Default']),
            'color': 'black',
            'weight': 1,
            'fillOpacity': 0.6,
        }

    try:
        # Use GeoPandas to read the file into a GeoDataFrame
        gdf_landuse = gpd.read_file(gpkg_path)

        # Pass the GeoDataFrame to Folium's GeoJson function
        folium.GeoJson(
            gdf_landuse,
            name='Land Use Zones',
            style_function=style_function,
            tooltip=folium.GeoJsonTooltip(fields=['use_type'], aliases=['Land Use:'])
        ).add_to(folium_map)
    except Exception as e:
        print(f"Error reading or processing GeoPackage file: {e}")
        print("Skipping the land use layer. Ensure the file exists and has a 'use_type' column.")

# ^-- THIS FUNCTION IS MODIFIED TO READ THE .GPKG FILE --^

def add_heatmap_layers(folium_map, df, top_n=5):
    """Adds a general heatmap and heatmaps for the top N most common species."""
    print("Generating heatmap layers...")
    df_agg = df.groupby(['LATITUDE', 'LONGITUDE'])['OBSERVATION COUNT'].sum().reset_index()
    heat_data_all = df_agg[['LATITUDE', 'LONGITUDE', 'OBSERVATION COUNT']].values.tolist()
    fg_all = folium.FeatureGroup(name="Heatmap: All Species", show=True).add_to(folium_map)
    HeatMap(heat_data_all, radius=15, blur=20).add_to(fg_all)

    top_species = df.groupby('COMMON NAME')['OBSERVATION COUNT'].sum().nlargest(top_n).index
    print(f"Top {top_n} species identified: {list(top_species)}")
    for species in top_species:
        df_species = df[df['COMMON NAME'] == species]
        df_species_agg = df_species.groupby(['LATITUDE', 'LONGITUDE'])['OBSERVATION COUNT'].sum().reset_index()
        heat_data_species = df_species_agg[['LATITUDE', 'LONGITUDE', 'OBSERVATION COUNT']].values.tolist()
        if not heat_data_species:
            continue
        fg_species = folium.FeatureGroup(name=f"Heatmap: {species}", show=False).add_to(folium_map)
        HeatMap(heat_data_species, radius=15, blur=20).add_to(fg_species)

# --- 4. MAIN EXECUTION ---
if __name__ == "__main__":
    df_birds = load_and_clean_data(CSV_PATH)
    
    print(f"Filtering observations within {RADIUS_KM} km of the airport...")
    df_birds['distance_km'] = vectorized_haversine(df_birds['LATITUDE'], df_birds['LONGITUDE'], AIRPORT_COORDS[0], AIRPORT_COORDS[1])
    df_filtered = df_birds[df_birds['distance_km'] <= RADIUS_KM].copy()
    print(f"Found {len(df_filtered)} observations within the radius.")
    
    if df_filtered.empty:
        print("No bird observations found within the specified radius. Exiting.")
        exit()

    # Create the map and add all layers
    map_final = create_base_map(AIRPORT_COORDS, RADIUS_KM)
    add_landuse_layer(map_final, GPKG_PATH) # <-- MODIFIED CALL
    add_heatmap_layers(map_final, df_filtered, top_n=TOP_N_SPECIES)
    
    # Finalize with a single LayerControl and save
    folium.LayerControl(collapsed=False).add_to(map_final)
    map_final.save(OUTPUT_HTML_FILE)
    
    print("-" * 50)
    print(f"Success! The interactive map has been saved as '{OUTPUT_HTML_FILE}'")
    print("-" * 50)

In [ ]:
import geopandas as gpd

# Make sure this path is correct
GPKG_PATH = "coimbatore_landUse.gpkg"

try:
    # Read the file into a GeoDataFrame
    gdf = gpd.read_file(GPKG_PATH)

    # Print the column names
    print("Columns found in the file:")
    print(list(gdf.columns))

    # Print the first few rows to see the data and columns together
    print("\nFirst 5 rows of data:")
    print(gdf.head())

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import fitz  # PyMuPDF

# Define the input PDF file and the desired output image name
pdf_path = "ExistingLanduseRuralUrbanA031May2025 - Compresed.pdf"
output_image_path = "base_map_high_res.png"

try:
    # Open the PDF file
    doc = fitz.open(pdf_path)
    
    # Load the first page (page 0)
    page = doc.load_page(0)
    
    # Set the zoom factor for high resolution. A zoom of 4x corresponds to 300 DPI.
    zoom_factor = 4.0  
    mat = fitz.Matrix(zoom_factor, zoom_factor)
    
    # Render the page to a pixmap (a pixel-based image)
    pix = page.get_pixmap(matrix=mat)
    
    # Save the pixmap as a PNG file
    pix.save(output_image_path)
    
    print(f"✅ Successfully rendered map to '{output_image_path}' with high resolution.")
    
    doc.close()

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import fitz  # PyMuPDF
import numpy as np
import json
import folium

# --- CONFIGURATION ---
PDF_PATH = "ExistingLanduseRuralUrbanA031May2025 - Compresed.pdf"
OUTPUT_GEOJSON = "extracted_landuse.geojson"
OUTPUT_MAP_HTML = "landuse_overlay_map.html"

# --- USER INPUT REQUIRED ---
# Replace these example values with the actual coordinates you find
# on your PDF map.
REFERENCE_POINTS = {
    "pdf_points": [
        [185, 230],  # PDF coordinates for Point 1 (X1, Y1)
        [1570, 1050] # PDF coordinates for Point 2 (X2, Y2)
    ],
    "geo_points": [
        [76.85, 11.15], # Geographic coordinates for Point 1 (Lon1, Lat1)
        [77.10, 10.95]  # Geographic coordinates for Point 2 (Lon2, Lat2)
    ]
}
# -------------------------

def create_affine_transform(ref_points):
    """Calculates the transformation matrix from PDF to Geographic coordinates."""
    pdf_pts = np.array(ref_points["pdf_points"])
    geo_pts = np.array(ref_points["geo_points"])
    
    # Pad matrices to solve for affine transformation
    A = np.vstack([
        [pdf_pts[0][0], pdf_pts[0][1], 1, 0, 0, 0],
        [0, 0, 0, pdf_pts[0][0], pdf_pts[0][1], 1],
        [pdf_pts[1][0], pdf_pts[1][1], 1, 0, 0, 0],
        [0, 0, 0, pdf_pts[1][0], pdf_pts[1][1], 1]
    ])
    
    b = np.array([geo_pts[0][0], geo_pts[0][1], geo_pts[1][0], geo_pts[1][1]])
    
    try:
        # Use least squares to find the best-fit transformation parameters
        transform, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        return transform
    except np.linalg.LinAlgError as e:
        print(f"❌ Could not solve for transformation: {e}")
        return None

def apply_transform(point, transform):
    """Applies the affine transformation to a single PDF point."""
    x, y = point
    lon = transform[0] * x + transform[1] * y + transform[2]
    lat = transform[3] * x + transform[4] * y + transform[5]
    return [lon, lat]

def extract_and_transform_vectors(pdf_path, transform):
    """
    Extracts vector paths from the PDF, handling sequential drawing commands,
    and applies the georeferencing transform.
    """
    doc = fitz.open(pdf_path)
    page = doc.load_page(0)
    paths = page.get_drawings()
    
    features = []
    for path in paths:
        current_point = None
        # Process each command (item) in a path sequence
        for item in path["items"]:
            command = item[0]
            
            if command == "m":  # "move to" command
                current_point = item[1] # Set the new current point
            
            elif command == "l": # "line to" command
                if current_point is None:
                    continue # Can't draw a line without a starting point
                
                start_pt = current_point
                end_pt = item[1]
                
                coords = [apply_transform(start_pt, transform), apply_transform(end_pt, transform)]
                features.append({
                    "type": "Feature",
                    "geometry": {"type": "LineString", "coordinates": coords},
                    "properties": {}
                })
                current_point = end_pt # The end of this line is the start of the next

            elif command == "re": # "rectangle" command
                rect = fitz.Rect(item[1])
                # Create a closed polygon (first and last points are the same)
                points = [rect.tl, rect.tr, rect.br, rect.bl, rect.tl]
                
                coords = [apply_transform(p, transform) for p in points]
                features.append({
                    "type": "Feature",
                    "geometry": {"type": "Polygon", "coordinates": [coords]}, # GeoJSON Polygons are nested in a list
                    "properties": {}
                })
                current_point = None # A closed shape has no "current point" after

            # Note: Curves ('c') are not handled in this version for simplicity.
            
    doc.close()
    return {"type": "FeatureCollection", "features": features}

# --- Main Execution ---
print("1. Calculating coordinate transformation...")
affine_transform = create_affine_transform(REFERENCE_POINTS)

if affine_transform is not None:
    print("✅ Transformation calculated successfully.")
    
    print("2. Extracting and georeferencing vector data from PDF...")
    geojson_data = extract_and_transform_vectors(PDF_PATH, affine_transform)
    
    with open(OUTPUT_GEOJSON, "w") as f:
        json.dump(geojson_data, f)
    print(f"✅ Vector data saved to '{OUTPUT_GEOJSON}'")

    print("3. Creating interactive map...")
    avg_lon = np.mean([p[0] for p in REFERENCE_POINTS["geo_points"]])
    avg_lat = np.mean([p[1] for p in REFERENCE_POINTS["geo_points"]])

    m = folium.Map(location=[avg_lat, avg_lon], zoom_start=12, tiles="OpenStreetMap")
    folium.GeoJson(geojson_data, name="Landuse Vectors").add_to(m)
    folium.LayerControl().add_to(m)

    m.save(OUTPUT_MAP_HTML)
    print(f"✅ Interactive map saved to '{OUTPUT_MAP_HTML}'. Open this file in a browser.")

In [ ]:
import folium
import rasterio
import numpy as np
from pyproj import Transformer
import warnings

# Suppress a common pyproj warning
warnings.filterwarnings("ignore", category=DeprecationWarning) 

# --- CONFIGURATION ---
geotiff_path = 'base_map_high_res_modified.tif' 
output_map_path = 'tif_on_openstreetmap.html'
# ---------------------

try:
    # Open the georeferenced raster file
    with rasterio.open(geotiff_path) as src:
        
        # 1. Get the map's bounds and CRS
        bounds = src.bounds
        native_crs = src.crs
        
        print(f"✅ Map opened. Original CRS is {native_crs}")
        
        # 2. Create a transformer to convert from the map's CRS to standard Lat/Lon
        # Web maps like OpenStreetMap use EPSG:4326 (WGS 84)
        transformer = Transformer.from_crs(native_crs, "EPSG:4326", always_xy=True)
        
        # 3. Transform the corner coordinates of the map
        bl = transformer.transform(bounds.left, bounds.bottom) # Bottom-left corner
        tr = transformer.transform(bounds.right, bounds.top)   # Top-right corner
        
        # Folium needs bounds in [[lat_min, lon_min], [lat_max, lon_max]] format
        reprojected_bounds = [[bl[1], bl[0]], [tr[1], tr[0]]]
        
        print(f"✅ Map bounds reprojected to Latitude/Longitude.")

        # 4. Read the image data and prepare it for display
        # We read the first 3 bands (Red, Green, Blue)
        image_data = src.read([1, 2, 3])
        
        # Transpose the array from (channel, height, width) to (height, width, channel)
        # This is the format Folium expects
        image_data = np.moveaxis(image_data, 0, -1)

        # 5. Create the interactive map
        # Calculate the center of the map for the initial view
        center_lat = (bl[1] + tr[1]) / 2
        center_lon = (bl[0] + tr[0]) / 2
        
        m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

        # 6. Add the TIF image as an overlay
        print("⏳ Adding image overlay to the map...")
        folium.raster_layers.ImageOverlay(
            image=image_data,
            bounds=reprojected_bounds,
            opacity=0.7, # You can adjust the transparency
            name="Land Use Map Overlay"
        ).add_to(m)

        folium.LayerControl().add_to(m)
        
        # 7. Save the map to an HTML file
        m.save(output_map_path)
        
        print(f"\n🎉 Success! Your interactive map is ready.")
        print(f"   -> Open the file '{output_map_path}' in your web browser.")

except FileNotFoundError:
    print(f"❌ Error: The file '{geotiff_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# --- 1. SETUP: Parameters and File Paths ---
# !!! IMPORTANT: UPDATE THESE PATHS !!!
CSV_PATH = "D:/Project_P1/data/ebird_observations.csv"
BASE_MAP_IMAGE_PATH = "base_map_high_res.png" # The land use map you provided
OUTPUT_IMAGE_FILE = "coimbatore_bird_heatmap_on_landuse_map.png"

AIRPORT_COORDS = (11.0296, 77.0434)
RADIUS_KM = 10

# --- 2. COORDINATE TRANSFORMATION SETUP ---
# !!! CRITICAL STEP: You MUST replace these placeholder values with your own measurements.
# Find the (pixel_x, pixel_y) from your image and the corresponding (lat, lon) from Google Maps.
TRANSFORMATION_POINTS = {
    'top_left': {
        'pixel': (306, 250),       # <-- YOUR Top-Left pixel coordinates here
        'geo': (11.200, 76.00)    # <-- YOUR Top-Left latitude, longitude here
    },
    'bottom_right': {
        'pixel': (11134, 9110),     # <-- YOUR Bottom-Right pixel coordinates here
        'geo': (10.900, 77.200)    # <-- YOUR Bottom-Right latitude, longitude here
    }
}

class CoordinateTransformer:
    """
    A class to handle the transformation from geographic (lat, lon) coordinates
    to image pixel (x, y) coordinates based on known reference points.
    """
    def __init__(self, points):
        if len(points) < 2:
            raise ValueError("At least two reference points are required for transformation.")
        
        geo_coords = np.array([[p['geo'][1], p['geo'][0]] for p in points.values()]) # [lon, lat]
        pixel_coords = np.array([[p['pixel'][0], p['pixel'][1]] for p in points.values()]) # [x, y]

        # Calculate scales (pixels per degree)
        delta_geo = geo_coords[1] - geo_coords[0]
        delta_pixel = pixel_coords[1] - pixel_coords[0]

        # Avoid division by zero if points are vertically or horizontally aligned
        if delta_geo[0] == 0 or delta_geo[1] == 0:
             raise ValueError("Reference points cannot be perfectly aligned vertically or horizontally in geo-coordinates.")

        self.x_scale = delta_pixel[0] / delta_geo[0]  # pixels per longitude degree
        self.y_scale = delta_pixel[1] / delta_geo[1]  # pixels per latitude degree

        # Calculate offsets
        self.x_offset = pixel_coords[0][0] - geo_coords[0][0] * self.x_scale
        self.y_offset = pixel_coords[0][1] - geo_coords[0][1] * self.y_scale
        
        print("Coordinate transformer initialized.")
        print(f"  X-Scale (px/lon): {self.x_scale:.2f}")
        print(f"  Y-Scale (px/lat): {self.y_scale:.2f}")

    def transform(self, lat: pd.Series, lon: pd.Series) -> (pd.Series, pd.Series):
        """Transforms lat/lon Series to pixel x/y Series."""
        x = lon * self.x_scale + self.x_offset
        y = lat * self.y_scale + self.y_offset
        return x, y

# --- 3. DATA PROCESSING (Mostly unchanged from your original script) ---

def load_and_clean_data(csv_path: str) -> pd.DataFrame:
    """Loads bird observation data, checks columns, and drops invalid rows."""
    print(f"Reading bird observation data from: {csv_path}")
    try:
        df = pd.read_csv(csv_path)
        required_columns = ['LATITUDE', 'LONGITUDE', 'COMMON NAME', 'OBSERVATION COUNT']
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"CSV must contain: {required_columns}")

        df.dropna(subset=['LATITUDE', 'LONGITUDE'], inplace=True)
        df['LATITUDE'] = pd.to_numeric(df['LATITUDE'], errors='coerce')
        df['LONGITUDE'] = pd.to_numeric(df['LONGITUDE'], errors='coerce')
        df['OBSERVATION COUNT'] = pd.to_numeric(df['OBSERVATION COUNT'], errors='coerce').fillna(1)
        df.dropna(subset=['LATITUDE', 'LONGITUDE', 'OBSERVATION COUNT'], inplace=True)
        
        print(f"Successfully loaded and cleaned {len(df)} observations.")
        return df
    except FileNotFoundError:
        print(f"Error: File not found at {csv_path}")
        exit()
    except Exception as e:
        print(f"An error occurred while reading the CSV: {e}")
        exit()

def vectorized_haversine(lats, lons, center_lat, center_lon):
    """Calculates great-circle distance in a vectorized way."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lats, lons, center_lat, center_lon])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# --- 4. MAIN EXECUTION & PLOTTING ---
if __name__ == "__main__":
    # Load and filter the data
    df_birds = load_and_clean_data(CSV_PATH)
    
    print(f"Filtering observations within {RADIUS_KM} km of the airport...")
    df_birds['distance_km'] = vectorized_haversine(df_birds['LATITUDE'], df_birds['LONGITUDE'], AIRPORT_COORDS[0], AIRPORT_COORDS[1])
    df_filtered = df_birds[df_birds['distance_km'] <= RADIUS_KM].copy()
    
    if df_filtered.empty:
        print("No bird observations found within the specified radius. Exiting.")
        exit()

    print(f"Found {len(df_filtered)} observations within the radius.")
    
    # Initialize the coordinate transformer
    transformer = CoordinateTransformer(TRANSFORMATION_POINTS)

    # Transform the filtered data's coordinates to pixels
    df_filtered['pixel_x'], df_filtered['pixel_y'] = transformer.transform(df_filtered['LATITUDE'], df_filtered['LONGITUDE'])

    # Load the base map image
    try:
        base_map_img = Image.open(BASE_MAP_IMAGE_PATH)
    except FileNotFoundError:
        print(f"Error: Base map image not found at {BASE_MAP_IMAGE_PATH}")
        exit()

    # --- Plotting using Matplotlib and Seaborn ---
    print("Generating plot...")
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(15, 15), dpi=150)

    # Display the base map image
    ax.imshow(base_map_img)

    # Overlay the heatmap using Seaborn's Kernel Density Estimate plot
    sns.kdeplot(
        x=df_filtered['pixel_x'],
        y=df_filtered['pixel_y'],
        weights=df_filtered['OBSERVATION COUNT'], # Weight the density by observation count
        cmap="Reds",  # A good colormap for heatmaps
        shade=True,   # Fill the contours
        alpha=0.5,    # Transparency
        ax=ax,
        n_levels=20,  # More levels for a smoother look
        bw_adjust=0.5 # Adjust bandwidth for smoothness
    )

    # Add a title and clean up the axes
    ax.set_title(f"Bird Observation Heatmap within {RADIUS_KM}km of Coimbatore Airport", fontsize=16)
    ax.set_axis_off() # Hide the pixel axes

    # Save the final image
    plt.tight_layout()
    plt.savefig(OUTPUT_IMAGE_FILE, bbox_inches='tight', pad_inches=0.1)
    
    print("-" * 50)
    print(f"Success! The heatmap image has been saved as '{OUTPUT_IMAGE_FILE}'")
    print("-" * 50)

In [ ]:
from PIL import Image

# --- [IMPORTANT] Handle high-resolution image ---
Image.MAX_IMAGE_PIPIXELS = None

# 1. Define the coordinates of your crop area from the original large image
#    (left, upper, right, lower)
CROP_X1 = 5700  # <-- X-coordinate of the TOP-LEFT corner of your land use map
CROP_Y1 = 3450  # <-- Y-coordinate of the TOP-LEFT corner
CROP_X2 = 8900 # <-- X-coordinate of the BOTTOM-RIGHT corner
CROP_Y2 = 5300  # <-- Y-coordinate of the BOTTOM-RIGHT corner

CROP_BOX = (CROP_X1, CROP_Y1, CROP_X2, CROP_Y2)

# 2. Open the original image, crop it, and save the new file
original_image = Image.open("bird_strike_risk_map.jpg")
cropped_image = original_image.crop(CROP_BOX)
cropped_image.save("landuse_map.png")

print("Successfully created 'landuse_map_cropped.png'")

In [ ]:
import cv2
import numpy as np

# --- 1. CONFIGURATION ---
BASE_MAP_IMAGE = "base_map_high_res.png"

# Colors (BGR)
COLOR_RED = (0, 0, 255)
COLOR_GREEN = (0, 255, 0)
COLOR_BLUE = (255, 0, 0)
COLOR_BLACK = (0, 0, 0)
COLOR_WHITE = (255, 255, 255)
COLOR_YELLOW = (0, 255, 255)
COLOR_PURPLE = (255, 0, 255)

# --- 2. GEOGRAPHIC TO PIXEL TRANSFORMATION POINTS (From your input) ---
# IMPORTANT: DOUBLE-CHECK THESE PIXEL AND GEO COORDINATES
# A small error here will throw off all subsequent plotting.
TRANSFORMATION_POINTS = {
    'p1': { # Top-Left of your reference box on the map
        'pixel': (6581, 4375),
        'geo': (11.0544, 77.064) # (Lat, Lon) for top-left
    },
    'p2': { # Bottom-Right of your reference box on the map
        'pixel': (5673, 3689),
        'geo': (11.0967, 77.0193) # (Lat, Lon) for bottom-right
    }
}

# --- 3. RUNWAY APPROACH GEOGRAPHIC COORDINATES (THESE NEED YOUR INPUT) ---
# RUNWAY_09_GEO = { # Landing East, Approach from West
#     'touchdown_geo': (11.0004, 76.9807), # Calculated West end of runway
#     'start_25km_geo': (10.8291, 76.8189)  # Calculated 25km start point
# }

RUNWAY_27_GEO = { # Landing West, Approach from East
    'touchdown_geo': (11.025, 77.066), # Confirmed East end of runway
    'start_25km_geo': (11.1632, 77.2117)  # Calculated 25km start point
}


# --- 4. DATA TABLES (from your previous excellent assessment) ---
# (I'm using a simplified version for demonstration. You can use your full tables.)

# RUNWAY 09 DATA (Landing East, Approach from West)
# RWY09_DATA = [
#     {"dist": 25, "threat": "HIGH THREAT", "species": "Black Kite, Indian Roller"},
#     {"dist": 20, "threat": "HIGH THREAT", "species": "Black Kite, Pariah Kite"},
#     {"dist": 15, "threat": "LOW THREAT", "species": "Rock Pigeon, House Crow"},
#     {"dist": 10, "threat": "LOW THREAT", "species": "Rock Pigeon, House Sparrow"},
#     {"dist": 5, "threat": "HIGH THREAT", "species": "Black Kite, Cattle Egret"},
#     {"dist": 1, "threat": "CRITICAL THREAT", "species": "Black Kite, Barn Owl"},
#     {"dist": 0, "threat": "CRITICAL THREAT", "species": "Rock Pigeon, House Crow"}, # Assuming touchdown is 0km
# ]

# RUNWAY 27 DATA (Landing West, Approach from East)
RWY27_DATA = [
    # {"dist": 25, "threat": "VERY HIGH THREAT", "species": "Black Kite, Brahminy Kite"},
    # {"dist": 20, "threat": "HIGH THREAT", "species": "Black Kite, House Crow"},
    {"dist": 15, "threat": "LOW-MODERATE THREAT", "species": "Rock Pigeon, Myna"},
    {"dist": 10, "threat": "LOW THREAT", "species": "Rock Pigeon, Myna"},
    {"dist": 5, "threat": "HIGH THREAT", "species": "Black Kite, Cattle Egret"},
    {"dist": 1, "threat": "CRITICAL THREAT", "species": "Black Kite, Short-eared Owl"},
    {"dist": 0, "threat": "CRITICAL THREAT", "species": "Rock Pigeon, House Crow"}, # Assuming touchdown is 0km
]


# --- HELPER FUNCTIONS ---

def geo_to_pixel(lat, lon, transform_points, img_width, img_height):
    """
    Converts geographic (lat, lon) to pixel coordinates (x, y) using two reference points.
    Assumes a linear transformation.
    """
    # Unpack transformation points
    p1_pixel_x, p1_pixel_y = transform_points['p1']['pixel']
    p1_geo_lat, p1_geo_lon = transform_points['p1']['geo']

    p2_pixel_x, p2_pixel_y = transform_points['p2']['pixel']
    p2_geo_lat, p2_geo_lon = transform_points['p2']['geo']

    # Calculate scaling factors (pixels per degree)
    # Note: Longitude scaling might vary with latitude, but for a small area, linear is okay.
    # We are inverting Y because pixel Y increases downwards, but Lat increases upwards.
    scale_x = (p2_pixel_x - p1_pixel_x) / (p2_geo_lon - p1_geo_lon)
    scale_y = (p2_pixel_y - p1_pixel_y) / (p2_geo_lat - p1_geo_lat) # Corrected: pixel Y increases with decreasing Lat

    # Calculate pixel coordinates
    pixel_x = p1_pixel_x + (lon - p1_geo_lon) * scale_x
    pixel_y = p1_pixel_y + (lat - p1_geo_lat) * scale_y
    
    # Ensure within image bounds (optional, but good for debugging)
    pixel_x = max(0, min(pixel_x, img_width - 1))
    pixel_y = max(0, min(pixel_y, img_height - 1))

    return int(pixel_x), int(pixel_y)


def get_threat_color(threat_str):
    if "CRITICAL" in threat_str.upper() or "VERY HIGH" in threat_str.upper():
        return COLOR_RED
    elif "HIGH" in threat_str.upper():
        return COLOR_YELLOW
    elif "MODERATE" in threat_str.upper():
        return COLOR_PURPLE
    else: # LOW THREAT
        return COLOR_GREEN

def draw_approach_path(image, start_geo, end_geo, data, color, transform_points):
    h, w, _ = image.shape

    start_pixel = geo_to_pixel(start_geo[0], start_geo[1], transform_points, w, h)
    end_pixel = geo_to_pixel(end_geo[0], end_geo[1], transform_points, w, h)

    cv2.line(image, start_pixel, end_pixel, color, 4, lineType=cv2.LINE_AA)
    cv2.circle(image, end_pixel, 15, COLOR_BLACK, -1) # Touchdown point
    cv2.circle(image, end_pixel, 12, COLOR_WHITE, -1)
    cv2.putText(image, "TD", (end_pixel[0]-10, end_pixel[1]+10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, COLOR_BLACK, 2)


    # Calculate points for markers along the path
    path_length_km = 25 # Assuming 25km approach
    num_markers = len(data) # Number of data points

    # Iterate through each data point to place markers and text
    for point_data in data:
        dist_from_td = point_data['dist']
        if dist_from_td == 0: # Handle touchdown point separately if needed
            continue # Already marked 'TD'

        # Calculate the interpolation factor (0 at start, 1 at end)
        # We need to map distance from TD to a position along the 25km path
        # If dist_from_td is 25, factor should be 0 (start). If 0, factor should be 1 (end).
        interp_factor = (path_length_km - dist_from_td) / path_length_km

        marker_pixel_x = int(start_pixel[0] + (end_pixel[0] - start_pixel[0]) * interp_factor)
        marker_pixel_y = int(start_pixel[1] + (end_pixel[1] - start_pixel[1]) * interp_factor)

        # Draw marker circle
        cv2.circle(image, (marker_pixel_x, marker_pixel_y), 8, COLOR_BLACK, -1) # Outer border
        cv2.circle(image, (marker_pixel_x, marker_pixel_y), 6, get_threat_color(point_data['threat']), -1) # Inner color

        # Add text label (distance from runway)
        text_offset_x = 10
        text_offset_y = -10
        if marker_pixel_x > w/2: # If on the right side, put text to the left
            text_offset_x = -100
        if marker_pixel_y < h/2: # If on top half, put text below
             text_offset_y = 20

        text = f"{dist_from_td}km"
        cv2.putText(image, text, (marker_pixel_x + text_offset_x, marker_pixel_y + text_offset_y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_BLACK, 2)

        # Add threat and species info in a small box near the marker
        # This part requires careful positioning to avoid clutter, especially with many points.
        # For a static image, it might be better to have a legend or hover-over effect.
        # For now, let's try to put it in a consistent spot relative to the marker.
        info_x_offset = 20
        info_y_offset = 5 # Starting from marker_pixel_y
        
        # Determine if text needs to go left/right or up/down
        text_anchor_x = marker_pixel_x + 15
        text_anchor_y = marker_pixel_y - 15
        
        # Basic logic to try and avoid edge
        if text_anchor_x + 250 > w: # Too close to right edge
            text_anchor_x = marker_pixel_x - 270 # Move left
        if text_anchor_y < 0: # Too close to top edge
            text_anchor_y = marker_pixel_y + 30 # Move down

        # Background rectangle for text
        text_lines = [
            f"Threat: {point_data['threat']}",
            f"Species: {point_data['species']}"
        ]
        line_height = 20
        max_text_width = 0
        for line in text_lines:
            (text_w, text_h), _ = cv2.getTextSize(line, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            if text_w > max_text_width:
                max_text_width = text_w
        
        cv2.rectangle(image, (text_anchor_x, text_anchor_y - line_height), 
                      (text_anchor_x + max_text_width + 10, text_anchor_y + len(text_lines) * line_height), 
                      COLOR_BLACK, -1)
        
        for j, line in enumerate(text_lines):
            cv2.putText(image, line, (text_anchor_x + 5, text_anchor_y + j * line_height),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, COLOR_WHITE, 1)


# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # Load the base map
    base_map = cv2.imread(BASE_MAP_IMAGE)
    if base_map is None:
        print(f"Error: Could not load the image at '{BASE_MAP_IMAGE}'. Make sure it's in the same folder.")
        exit()

    img_height, img_width, _ = base_map.shape
    
    # Create a copy to draw on
    output_map = base_map.copy()

    # print("Drawing Runway 09 Approach (Landing East)...")
    # # Runway 09 (Landing East, Approach from West)
    # draw_approach_path(
    #     output_map,
    #     RUNWAY_09_GEO['start_25km_geo'],
    #     RUNWAY_09_GEO['touchdown_geo'],
    #     RWY09_DATA,
    #     COLOR_BLUE, # Blue for Runway 09 path
    #     TRANSFORMATION_POINTS
    # )

    print("Drawing Runway 27 Approach (Landing West)...")
    # Runway 27 (Landing West, Approach from East)
    draw_approach_path(
        output_map,
        RUNWAY_27_GEO['start_25km_geo'],
        RUNWAY_27_GEO['touchdown_geo'],
        RWY27_DATA,
        COLOR_RED, # Red for Runway 27 path
        TRANSFORMATION_POINTS
    )

    # Save the output image
    output_filename = "bird_strike_risk_map.jpg"
    cv2.imwrite(output_filename, output_map)
    print(f"✅ Static risk map saved as '{output_filename}'")
    
    # Optional: Display the image
    # cv2.imshow("Bird Strike Risk Map", output_map)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

## Review-03


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from PIL import Image

# --- [NEW] Increase the pixel limit for large images to prevent DecompressionBombError ---
Image.MAX_IMAGE_PIXELS = None

# --- 1. SETUP: Parameters and File Paths ---
# Make sure these paths are correct for your Jupyter environment
CSV_PATH = "D:/Project_P1/data/ebird_observations.csv"
BASE_MAP_IMAGE_PATH = "base_map_high_res.png" 

AIRPORT_COORDS = (11.0296, 77.0434)
RADIUS_KM = 10

TRANSFORMATION_POINTS = {
    'top_left': {
        'pixel': (6581,4375),        # <-- Your new pixel values from the script
        'geo': (11.0544, 77.064)     # <-- Your new geo values from Google Maps
    },
    'bottom_right': {
        'pixel': (5673,3689),       # <-- Your second new pixel values
        'geo': (11.0967, 77.0193)     # <-- Your second new geo values
    }
}

# --- 3. HELPER CLASSES AND FUNCTIONS ---
class CoordinateTransformer:
    def __init__(self, points):
        if len(points) < 2: raise ValueError("At least two reference points are required.")
        geo_coords = np.array([[p['geo'][1], p['geo'][0]] for p in points.values()])
        pixel_coords = np.array([[p['pixel'][0], p['pixel'][1]] for p in points.values()])
        delta_geo = geo_coords[1] - geo_coords[0]
        delta_pixel = pixel_coords[1] - pixel_coords[0]
        if delta_geo[0] == 0 or delta_geo[1] == 0: raise ValueError("Reference points cannot be perfectly aligned.")
        self.x_scale = delta_pixel[0] / delta_geo[0]
        self.y_scale = delta_pixel[1] / delta_geo[1]
        self.x_offset = pixel_coords[0][0] - geo_coords[0][0] * self.x_scale
        self.y_offset = pixel_coords[0][1] - geo_coords[0][1] * self.y_scale
    def transform(self, lat: pd.Series, lon: pd.Series):
        x = lon * self.x_scale + self.x_offset
        y = lat * self.y_scale + self.y_offset
        return x, y

def load_and_clean_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df.dropna(subset=['LATITUDE', 'LONGITUDE'], inplace=True)
    df['LATITUDE'] = pd.to_numeric(df['LATITUDE'], errors='coerce')
    df['LONGITUDE'] = pd.to_numeric(df['LONGITUDE'], errors='coerce')
    df['OBSERVATION COUNT'] = pd.to_numeric(df['OBSERVATION COUNT'], errors='coerce').fillna(1)
    df.dropna(subset=['LATITUDE', 'LONGITUDE', 'OBSERVATION COUNT'], inplace=True)
    return df

def vectorized_haversine(lats, lons, center_lat, center_lon):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lats, lons, center_lat, center_lon])
    dlon = lon2 - lon1; dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# --- 4. MAIN EXECUTION & PLOTTING ---
# Load and filter data
df_birds = load_and_clean_data(CSV_PATH)
df_birds['distance_km'] = vectorized_haversine(df_birds['LATITUDE'], df_birds['LONGITUDE'], AIRPORT_COORDS[0], AIRPORT_COORDS[1])
df_filtered = df_birds[df_birds['distance_km'] <= RADIUS_KM].copy()

if df_filtered.empty:
    print("No bird observations found within the specified radius. Exiting.")
else:
    transformer = CoordinateTransformer(TRANSFORMATION_POINTS)
    df_filtered['pixel_x'], df_filtered['pixel_y'] = transformer.transform(df_filtered['LATITUDE'], df_filtered['LONGITUDE'])

    img = Image.open(BASE_MAP_IMAGE_PATH)
    img_width, img_height = img.size

    # Define the list of species for the dropdown
    species_list = [
        'All Species', 'Barn Swallow', 'Black Kite', 'Eurasian Kestrel', 'House Crow',
        'Rose-ringed Parakeet', 'Red-wattled Lapwing', 'Common Myna', 'Rock Pigeon',
        'Yellow-wattled Lapwing', 'Indian Peafowl', 'Eastern Cattle-Egret',
        'Indian Roller', 'Black Drongo'
    ]

    print("Generating Plotly figure with species selector...")
    fig = go.Figure()

    # Create a trace for each species in the list
    for species in species_list:
        if species == 'All Species':
            df_species = df_filtered
        else:
            df_species = df_filtered[df_filtered['COMMON NAME'] == species]

        is_visible = (species == 'All Species')

        # Add the heatmap contour trace WITHOUT a filled background
        fig.add_trace(go.Histogram2dContour(
            x=df_species['pixel_x'],
            y=df_species['pixel_y'],
            # Removed 'z' parameter to prevent solid background fill
            # Using colorscale that goes from transparent to red
            colorscale=[[0, 'rgba(255,255,255,0)'], [0.1, 'rgba(255,230,230,0.5)'], [1, 'rgb(255,0,0)']],
            showscale=False,
            name=species,
            visible=is_visible,
            # Configure the contours for better visibility
            contours=dict(
                coloring='lines', # 'fill', 'heatmap', or 'lines' - 'lines' for contours only
                showlabels=False, # Set to True if you want contour values on lines
                start=1, # Start contours from a count of 1
                end=df_species['OBSERVATION COUNT'].max() * 0.8, # Adjust max contour level
                size=1 # Distance between contours
            ),
            line_width=1.5 # Thickness of the contour lines
        ))

        # Add the corresponding hover trace for the species (UNTOUCHED)
        fig.add_trace(go.Scatter(
            x=df_species['pixel_x'],
            y=df_species['pixel_y'],
            mode='markers',
            marker=dict(color='rgba(0,0,0,0)'),
            hoverinfo='text',
            text=df_species.apply(lambda r: f"<b>{r['COMMON NAME']}</b><br>Count: {r['OBSERVATION COUNT']}", axis=1),
            name=f"{species} Hover",
            visible=is_visible
        ))

    # Add the land use map as a background image
    fig.add_layout_image(dict(source=img, xref="x", yref="y", x=0, y=0, sizex=img_width, sizey=img_height, sizing="stretch", opacity=0.8, layer="below"))

    # Create the dropdown menu buttons
    buttons = []
    for i, species in enumerate(species_list):
        visibility_mask = [False] * (len(species_list) * 2)
        visibility_mask[i*2] = True      # Show the heatmap trace
        visibility_mask[i*2 + 1] = True  # Show the hover trace
        buttons.append(dict(
            method='restyle',
            label=species,
            args=[{'visible': visibility_mask}]
        ))

    # Update layout with the new dropdown menu
    fig.update_layout(
        title=f"Interactive Bird Observation Heatmap within {RADIUS_KM}km of Airport",
        xaxis=dict(range=[0, img_width], showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(range=[img_height, 0], showgrid=False, zeroline=False, showticklabels=False),
        # Adjust width/height as needed for your notebook display
        width=1000, 
        height=800,
        margin=dict(l=10, r=10, t=60, b=10),
        hovermode='closest',
        updatemenus=[dict(
            active=0, buttons=buttons, direction="down",
            pad={"r": 10, "t": 10}, showactive=True,
            x=0.01, xanchor="left", y=1.1, yanchor="top"
        )]
    )
    
    # --- [MODIFIED] Display the figure directly in the notebook ---
    fig.show()
    # # Save the figure as an interactive HTML file
    # output_html_path = "bird_heatmap_interactive.html"
    # fig.write_html("plot.html", include_plotlyjs='cdn')
    # print(f"✅ Figure saved to: {output_html_path}")